# Idea 1, the rule fixed in advance, on a fresh photo set
Rule (fixed before looking at this set, from Idea1c): confidence >= 0.05, tolerance 20 m, answer at the best place when z >= 10 and every place farther than 2 sigma is at least 8 lower; applied to photos the FIRST PASS (6 standard queries, bar 8 or two queries agreeing) skips. 
Set: `lowconf_<TAG>` (env TAG), a low-threshold first-pass dump of photos that no earlier test used (NUDT valid split, photos the valid1212 draw did not use).
Reading: score of the first pass alone against first pass + rescue (no escalation in either), rescued photos, wrong answers.
Falsification (planned): dead if the rescue adds any wrong answer, or gains nothing.

In [1]:
import os, sys, math
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"
from train.checks import replay
from fine_matching.loftr_matcher import reliable_mask
from image_processing.map_levels import level_gsd
TAG = os.environ["TAG"]
low = replay.load_set(T7 / "rich_dump" / ("lowconf_" + TAG), with_matches=True)
print(len(low), "photos")

150 photos


In [2]:
SIZE = 224
def z_of(photo, level, c, t):
    """For every standard query and candidate: (z, x, y, inliers) with z the excess count of aligned matches over chance."""
    gsd = level_gsd(level); out = []
    for qi in replay.query_ids(photo, replay.STANDARD):
        q = photo["queries"][qi]; reliable = reliable_mask(q["inside"], q["border_px"])
        for k, m in enumerate(q["matches"]):
            if m is None or q["peaks"][k][1] > 1e8:
                continue
            a, b, conf = m["points_a"], m["points_b"], m["confidence"]
            cols = np.clip(np.round(a[:, 0]).astype(int), 0, reliable.shape[1] - 1); rows = np.clip(np.round(a[:, 1]).astype(int), 0, reliable.shape[0] - 1)
            chosen = reliable[rows, cols] & (conf >= c)
            n = int(chosen.sum()); gap = np.hypot(b[chosen, 0] - a[chosen, 0], b[chosen, 1] - a[chosen, 1]) * gsd
            near = int((gap <= t).sum()); chance = n * min(1.0, math.pi * (t / gsd) ** 2 / (SIZE * SIZE))
            out.append(((near - chance) / math.sqrt(chance + 1.0), q["peaks"][k][1] + q["shifts"][k][0], q["peaks"][k][2] + q["shifts"][k][1], float(q["inliers"][k]), qi))
    return out


In [3]:
def rescue(cands, photo, thr, margin, radius_sigma=2.0):
    """Answer at the candidate with the highest z if z >= thr and no candidate farther than radius_sigma from it has z within `margin` of it."""
    if not cands:
        return None
    z = np.array([c[0] for c in cands]); best = int(np.argmax(z))
    if z[best] < thr:
        return None
    far = [c[0] for c in cands if math.hypot(c[1] - cands[best][1], c[2] - cands[best][2]) / photo["sigma"] > radius_sigma]
    if far and max(far) > z[best] - margin:
        return None
    return cands[best][1], cands[best][2]



In [4]:
C, T, THR, MARGIN = 0.05, 20.0, 10.0, 8.0
rows = []
for pid, photo in low.items():
    first = replay.decide_current(photo, escalate=False)
    points, wrong, rescued = first["points"], int(first["answered"] and first["err_sigma"] > 8), 0
    if not first["answered"]:
        place = rescue(z_of(photo, photo["level"], C, T), photo, THR, MARGIN)
        if place is not None:
            err = replay.error_sigma(photo, *place); points, wrong, rescued = replay.points(err), int(err > 8), 1
    rows.append(dict(photo=pid, height=photo["height"], first_points=first["points"], first_wrong=int(first["answered"] and first["err_sigma"] > 8), first_stage=first["stage"], points=points, wrong=wrong, rescued=rescued))
R = pd.DataFrame(rows); R.to_csv(T7 / ("idea1d_fixed_rule_%s.csv" % TAG), index=False)
n = len(R)
print("photos %d | first pass %.2f -> with rescue %.2f of 60 | skipped by first pass %d, rescued %d | wrong: first pass %d, with rescue %d" %
      (n, R.first_points.sum() / n * 6, R.points.sum() / n * 6, int((R.first_stage == "skip").sum()), int(R.rescued.sum()), int(R.first_wrong.sum()), int(R.wrong.sum())))
print(R[R.rescued == 1][["photo", "height", "points"]].to_string(index=False))

photos 150 | first pass 52.08 -> with rescue 51.64 of 60 | skipped by first pass 13, rescued 3 | wrong: first pass 2, with rescue 3
 photo  height  points
   547 197.211       3
   602 208.322       6
   985 301.143     -20
